# Baseline-selection inspection and evaluation convention

**Research question.** What structure in FinQA should the M3 baseline exploit, and does the project's
current evaluation measure what the official benchmark measures?

Notebooks 06–08 scored 0–3.7% on dev under the display-answer comparator, with 0/492 percentage
answers correct. Before choosing a stronger baseline, this notebook inspects **train** annotations
(programs, gold evidence, answers), which is legitimate supervision for design decisions. Dev is used
only (a) to check that the safe executor reproduces `exe_ans` from gold programs and (b) to re-score the
already-fixed dev prediction artifacts of Notebooks 06–08. The official test split is not loaded.

In [1]:
from __future__ import annotations

import json
import re
import tomllib
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from finevidence.data import build_structured_input, load_finqa_split, parse_finqa_program
from finevidence.evaluation import compare_display_answers, compare_execution, gold_recall_at_k
from finevidence.reasoning import execute_program, official_str_to_num

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
config = tomllib.loads(
    (PROJECT_ROOT / "configs" / "baseline_selection_inspection.toml").read_text()
)
RAW_DIR = PROJECT_ROOT / config["dataset_root"]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
train = load_finqa_split(RAW_DIR / f"{config['train_split']}.json")
dev = load_finqa_split(RAW_DIR / f"{config['evaluation_split']}.json")
print({"train": len(train), "dev": len(dev), "test_loaded": False})

{'train': 6251, 'dev': 883, 'test_loaded': False}


## 1. Program structure (train)

If a small number of operation templates covers most programs, a generator over a constrained program
vocabulary is a reasonable target, and template frequency bounds what a template classifier could reach.

In [2]:
parsed = [parse_finqa_program(record["qa"]["program"]) for record in train]
assert all(result.is_valid for result in parsed)
templates = Counter("|".join(step.operation for step in result.steps) for result in parsed)
steps = Counter(len(result.steps) for result in parsed)

template_table = pd.DataFrame(templates.most_common(12), columns=["template", "train_records"])
template_table["cumulative_share"] = (template_table["train_records"].cumsum() / len(train)).round(
    4
)
display(
    pd.Series(dict(sorted(steps.items())), name="train_records")
    .rename_axis("program_steps")
    .to_frame()
)
display(template_table)
print({"distinct_templates": len(templates)})

,train_records
program_steps,
1,3717
2,2013
3,331
4,94
5,90
6,6


,template,train_records,cumulative_share
0,divide,2128,0.3404
1,subtract|divide,1327,0.5527
2,subtract,884,0.6941
3,add,251,0.7343
4,add|divide,185,0.7639
5,multiply,159,0.7893
6,add|add,142,0.8120
7,multiply|divide,133,0.8333
8,greater,110,0.8509
9,add|add|divide,108,0.8682


{'distinct_templates': 119}


## 2. Where do program operands come from? (train)

A FinQANet-style generator copies numbers from retrieved evidence rather than generating digits. That is
only appropriate if program literals actually appear in the context. Numbers are compared as absolute
values rounded to six decimals, because FinQA programs often drop the sign or accounting parentheses
shown in the report.

In [3]:
NUMBER = re.compile(r"-?\d+(?:,\d{3})*(?:\.\d+)?|-?\.\d+")
LITERAL = re.compile(r"-?(?:\d+(?:\.\d+)?|\.\d+)%?")


def numbers_in(text: str) -> set[float]:
    return {round(abs(float(match.replace(",", ""))), 6) for match in NUMBER.findall(str(text))}


def evidence_segments(record) -> dict[str, str]:
    # Keys follow the verified gold_inds convention: table_n is raw row n; text_n indexes pre+post text.
    source = build_structured_input(record)
    table = source["table"]
    header = [str(cell) for cell in table[0]] if table else []
    segments = {
        f"table_{index}": " ".join(f"{h} {cell}" for h, cell in zip(header, row, strict=False))
        for index, row in enumerate(table)
    }
    for index, sentence in enumerate(source["pre_text"] + source["post_text"]):
        segments[f"text_{index}"] = sentence
    return segments


kinds, provenance = Counter(), Counter()
for record, result in zip(train, parsed, strict=True):
    segments = evidence_segments(record)
    gold_numbers = set().union(set(), *(numbers_in(v) for v in record["qa"]["gold_inds"].values()))
    for step in result.steps:
        for argument, kind in zip(step.arguments, step.argument_kinds, strict=True):
            kinds[kind] += 1
            if not LITERAL.fullmatch(argument):
                continue
            value = round(abs(float(argument.rstrip("%"))), 6)
            sources = {
                key.split("_")[0] for key, text in segments.items() if value in numbers_in(text)
            }
            provenance["literals"] += 1
            provenance["in_gold_evidence"] += value in gold_numbers
            provenance["in_any_context"] += bool(sources)
            provenance[
                "table_only"
                if sources == {"table"}
                else "text_only"
                if sources == {"text"}
                else "table_and_text"
                if sources
                else "not_found"
            ] += 1
display(pd.Series(kinds, name="train_arguments").to_frame())
display(pd.Series(provenance, name="train_numeric_literals").to_frame())
print({"records_using_named_constants": sum("const_" in r["qa"]["program"] for r in train)})

,train_arguments
numeric_literal,14204
intermediate_reference,3354
named_constant,1230
table_label,204
none,204


,train_numeric_literals
literals,14210
in_gold_evidence,14188
in_any_context,14190
text_only,3782
table_only,8847
table_and_text,1561
not_found,20


{'records_using_named_constants': 901}


## 3. Gold evidence size and context length (train)

Context length decides whether an encoder can read the whole report or needs a retriever first.
Lengths are whitespace tokens of question + table rows + text, a lower bound on sub-word tokens.

In [4]:
gold_counts = pd.Series([len(r["qa"]["gold_inds"]) for r in train])
source_mix = Counter(
    "+".join(sorted({key.split("_")[0] for key in r["qa"]["gold_inds"]})) for r in train
)
lengths = pd.Series(
    [
        len(r["qa"]["question"].split())
        + sum(len(s.split()) for s in evidence_segments(r).values())
        for r in train
    ]
)
budget = config["encoder_token_budget"]
display(gold_counts.describe().round(2).rename("gold_facts_per_record").to_frame())
display(pd.Series(source_mix, name="train_records").rename_axis("gold_source_mix").to_frame())
print(
    {
        "context_tokens_median": int(lengths.median()),
        "context_tokens_p90": int(lengths.quantile(0.9)),
        f"records_over_{budget}": int((lengths > budget).sum()),
        "gold_table_0_references": sum("table_0" in r["qa"]["gold_inds"] for r in train),
    }
)

,gold_facts_per_record
count,6251.00
mean,1.71
std,0.85
min,1.00
25%,1.00
50%,2.00
75%,2.00
max,9.00


,train_records
gold_source_mix,
text,1458
table+text,871
table,3922


{'context_tokens_median': 762, 'context_tokens_p90': 1084, 'records_over_512': 5398, 'gold_table_0_references': 39}


## 4. Is TF-IDF retrieval the bottleneck? (train)

Within each record, segments are ranked by train-fitted TF-IDF similarity to the question and compared
with `gold_inds`. This is a diagnostic of lexical retrieval quality, not a dev result.

In [5]:
train_segments = [evidence_segments(record) for record in train]
vectorizer = TfidfVectorizer(
    ngram_range=(config["ngram_min"], config["ngram_max"]),
    max_features=config["max_features"],
    sublinear_tf=True,
).fit(
    [text for segs in train_segments for text in segs.values()]
    + [r["qa"]["question"] for r in train]
)

recall_rows = []
for record, segments in zip(train, train_segments, strict=True):
    keys = list(segments)
    gold = [key for key in record["qa"]["gold_inds"] if key in segments]
    if not gold:
        continue
    scores = (
        vectorizer.transform(segments.values()) @ vectorizer.transform([record["qa"]["question"]]).T
    )
    ranked = [keys[i] for i in np.argsort(-scores.toarray().ravel(), kind="stable")]
    for k in config["recall_k"]:
        recall = gold_recall_at_k(ranked, gold, k)
        recall_rows.append(
            {
                "k": k,
                "any_gold": recall.any_gold,
                "all_gold": recall.all_gold,
                "multi_fact": len(gold) > 1,
            }
        )
recall_frame = pd.DataFrame(recall_rows)
display(recall_frame.groupby("k")[["any_gold", "all_gold"]].mean().round(4))
display(
    recall_frame[recall_frame.multi_fact]
    .groupby("k")[["all_gold"]]
    .mean()
    .round(4)
    .rename(columns={"all_gold": "all_gold_multi_fact_only"})
)

,any_gold,all_gold
k,,
1,0.5698,0.2779
3,0.8290,0.5383
5,0.8970,0.6633
10,0.9645,0.8194


,all_gold_multi_fact_only
k,
1,0.0000
3,0.2846
5,0.4605
10,0.6971


## 5. Evaluation convention: display answer vs. official execution accuracy

The official evaluator at the pinned FinQA commit (`code/evaluate/evaluate.py`) executes the predicted
program, rounds a numeric result to five decimals, and requires exact equality with `exe_ans`; all records
stay in the denominator. `finevidence.reasoning.execute_program` reimplements those value rules on top of
the non-executing parser (no `eval`).

**Executor check.** Executing each *gold* program must reproduce `exe_ans`. Running dev gold programs
here validates the executor only; nothing is tuned on them.

In [6]:
executor_check = {}
for name, records in {"train": train, "dev": dev}.items():
    statuses = Counter(
        compare_execution(
            execute_program(r["qa"]["program"], r["table"]).value, r["qa"]["exe_ans"]
        ).status.value
        for r in records
    )
    executor_check[name] = dict(statuses)
display(pd.DataFrame(executor_check).fillna(0).astype(int))

,train,dev
correct,6251,883


**Display-comparator ceiling (train).** If a system produced the *gold* executed value, how often would
the display-answer comparator from Notebook 05 accept it? Percentage answers are tried both as the raw ratio
and multiplied by 100.

In [7]:
ceiling = Counter()
for record in train:
    answer, executed = record["qa"]["answer"], record["qa"]["exe_ans"]
    if not answer.strip():
        ceiling["empty_display_reference"] += 1
        continue
    if not isinstance(executed, float):
        ceiling["non_numeric_exe_ans"] += 1
        continue
    is_percent = answer.strip().endswith("%")
    scaled = f"{executed * 100!r}%" if is_percent else repr(executed)
    status = compare_display_answers(scaled, answer).status.value
    ceiling[f"{'percent' if is_percent else 'number'}_gold_value_{status}"] += 1
    if is_percent:
        ceiling[
            f"percent_raw_ratio_{compare_display_answers(repr(executed), answer).status.value}"
        ] += 1
display(pd.Series(ceiling, name="train_records").sort_index().to_frame())

,train_records
empty_display_reference,48
non_numeric_exe_ans,120
number_gold_value_correct,1720
number_gold_value_incorrect,883
percent_gold_value_correct,483
percent_gold_value_incorrect,2997
percent_raw_ratio_incorrect,3480


## 6. Re-scoring Notebook 06–08 dev predictions under execution accuracy

The stored dev predictions are fixed artifacts; only the scoring changes. Each prediction string is
converted with the official token rules (`$` and spaces stripped, `%` divided by 100). The denominator is
all 883 dev records, following the official convention.

In [8]:
artifacts = {
    "06 lexical value selection": "lexical_value_baseline_dev_predictions.jsonl",
    "07 question-conditioned selection": "question_conditioned_value_selection_dev_predictions.jsonl",
    "08 conservative arithmetic": "conservative_arithmetic_dev_predictions.jsonl",
}
display_correct_key = {
    "06 lexical value selection": "comparison_status",
    "07 question-conditioned selection": "comparison_status",
    "08 conservative arithmetic": "status",
}
dev_by_id = {record["id"]: record for record in dev}
rescored = []
for name, filename in artifacts.items():
    rows = [json.loads(line) for line in (PROCESSED_DIR / filename).read_text().splitlines()]
    assert len(rows) == len(dev) and {row["id"] for row in rows} == set(dev_by_id)
    correct = 0
    for row in rows:
        raw = row["prediction"]
        value = None
        if raw is not None:
            token = str(raw).replace("$", "").replace(" ", "")
            value = token.lower() if token.lower() in {"yes", "no"} else official_str_to_num(token)
        correct += compare_execution(value, dev_by_id[row["id"]]["qa"]["exe_ans"]).is_correct
    rescored.append(
        {
            "run": name,
            "display_correct_of_871": sum(
                row[display_correct_key[name]] == "correct" for row in rows
            ),
            "execution_correct_of_883": correct,
            "execution_accuracy": round(correct / len(dev), 6),
        }
    )
display(pd.DataFrame(rescored))

,run,display_correct_of_871,execution_correct_of_883,execution_accuracy
0,06 lexical value selection,3,5,0.005663
1,07 question-conditioned selection,3,6,0.006795
2,08 conservative arithmetic,32,35,0.039638



## Findings and decision

Interpretation of the outputs above (train unless stated):

- **Programs are short and template-concentrated.** 3,717/6,251 programs have one step and 5,730 have at most two; 119 distinct operation templates exist, and the top ten cover 86.8%. A generator over a constrained program vocabulary is a well-scoped target.
- **Operands are copyable.** 14,190/14,210 numeric literals appear in the record's context and 14,188 in its gold evidence (sign ignored). 901 records also need named constants. This supports a generator that *points to* numbers in retrieved evidence plus a small constant/operation vocabulary, rather than generating digits.
- **Retrieval is required and multi-fact recall is the weak spot.** Median context is 762 whitespace tokens and 5,398 records exceed 512, so an encoder cannot read the full record. Train-fitted TF-IDF ranks *some* gold fact in the top 3 for 82.9% of records, but *all* gold facts for only 53.8%, and for only 28.5% of multi-fact records. A trained retriever should be judged on all-gold recall.
- **The display comparator was the dominant cause of near-zero scores.** The executor reproduces `exe_ans` for 6,251/6,251 train and 883/883 dev gold programs. Yet the gold executed value passes the display comparator for only 2,203 of 6,083 numeric-`exe_ans` train records (1,720 numbers, 483 percentages), because display answers are rounded and percentages are scaled by 100. Raw ratios never match percentage answers (0/3,480).
- **Re-scored under official execution accuracy** (denominator 883): lexical 5 (0.57%), question-conditioned 6 (0.68%), conservative arithmetic 35 (3.96%). The ordering is unchanged; the earlier runs remain weak lower bounds.

**Decision.** Adopt official execution accuracy as the primary M3 answer metric (display-answer accuracy is kept as a secondary, documented metric), report retrieval as any/all-gold recall@k, and build a FinQANet-style two-stage baseline: a fine-tuned cross-encoder retriever trained on train `gold_inds`, followed by a program generator that copies operands from retrieved facts and is executed by `execute_program`. Program accuracy (symbolic equivalence) still needs an implementation before it can be reported. The plan is in `docs/finqanet-baseline-plan.md`.
